# Practice Exercise 4



## Step 1: Identify the Task

The task is to develop a knowledge-based self-driving trailer agent capable of navigating the Kenyan highway network from Nairobi to Kisumu.

Goal: Reach Kisumu safely along the lowest-distance path ($370\text{ km}$ via $\text{Nairobi} \rightarrow \text{Naivasha} \rightarrow \text{Nakuru} \rightarrow \text{Molo} \rightarrow \text{Kericho} \rightarrow \text{Kisumu}$), while following road safety rules, protecting cargo and adapting to dynamic road hazards.

## Step 2: Assemble Relevant Knowledge

The knowledge base incorporates:
* **PEAS & Environment Properties (from Exercise 3):** Partially observable, stochastic, sequential, dynamic, continuous, and multi-agent on the physical highway; fully observable, deterministic, sequential, static, discrete, and single-agent on the route map.
* **Directed Highway Network & Distances (km):** Road segments connecting Nairobi, Ruiru, Thika, Nyeri, Nyahururu, Naivasha, Nakuru, Molo, Kericho, Kisumu, Narok, Bomet, Eldoret, Kapsabet, Kisii, Eldama Ravine, Embu, Meru, Nanyuki, Webuye, and Kakamega.
* **Safety & Driving Rules:** Operational responses to road conditions (Clear, Fog, Potholes, Traffic Jam, Roadblock, Heavy Rain).

**APIs / Web Scraping Declaration:** No external APIs or web scrapers are required to execute the offline knowledge base. However, an optional ingestion hook (`fetch_external_road_advisories()`) is included in Step 5 to demonstrate merging live traffic or weather advisories into the KB.

## Step 3: Define the Vocabulary

* **Predicates / Relations (Python structures & functions):**
  * `Road(origin, destination, distance_km)` — directed highway segment and distance in km.
  * `OptimalPath(start, goal)` — computes lowest cumulative cost $g(n)$ using Uniform Cost Search.
  * `RoadConditionRule(condition)` — maps road hazards/signals to safe trailer actions.
  * `CanProceed(state)` — evaluates safety and goal status (`current_town == "Kisumu"`).
* **Constants:**
  * Towns: `"Nairobi"`, `"Ruiru"`, `"Thika"`, `"Nyeri"`, `"Nyahururu"`, `"Naivasha"`, `"Nakuru"`, `"Molo"`, `"Kericho"`, `"Kisumu"`, `"Narok"`, `"Bomet"`, `"Eldoret"`, `"Kapsabet"`, `"Kisii"`, `"Eldama Ravine"`, `"Embu"`, `"Meru"`, `"Nanyuki"`, `"Webuye"`, `"Kakamega"`.
  * Road/Hazard States: `"Clear"`, `"Fog"`, `"Potholes"`, `"Traffic Jam"`, `"Roadblock"`, `"Heavy Rain"`.

## Step 4: Encode General Knowledge about the Domain 

In [1]:
import heapq

# Directed highway graph from Exercise 3: {town: [(next_town, distance_km), ...]}
highway_kb = {
    "Nairobi": [("Ruiru", 20), ("Naivasha", 90)],
    "Ruiru": [("Thika", 25)],
    "Thika": [("Nyeri", 110), ("Embu", 85)],
    "Embu": [("Meru", 75), ("Thika", 85)],
    "Meru": [("Nanyuki", 75)],
    "Nanyuki": [("Nyeri", 55)],
    "Nyeri": [("Nairobi", 115), ("Nyahururu", 80), ("Nanyuki", 55)],
    "Nyahururu": [("Nakuru", 105), ("Eldama Ravine", 70), ("Nyeri", 80)],
    "Eldama Ravine": [("Eldoret", 90), ("Nyahururu", 70)],
    "Nakuru": [
        ("Naivasha", 65),
        ("Nyahururu", 105),
        ("Eldama Ravine", 70),
        ("Eldoret", 155),
        ("Molo", 50),
        ("Nyeri", 185),
    ],
    "Naivasha": [("Nakuru", 65), ("Narok", 100), ("Nairobi", 90)],
    "Narok": [("Naivasha", 100), ("Bomet", 85)],
    "Bomet": [("Narok", 85), ("Kericho", 60), ("Kisii", 75)],
    "Kisii": [("Bomet", 75), ("Kisumu", 115)],
    "Kericho": [("Bomet", 60), ("Molo", 65), ("Kisumu", 100)],
    "Molo": [("Nakuru", 50), ("Kericho", 65)],
    "Eldoret": [("Webuye", 75), ("Kapsabet", 40), ("Eldama Ravine", 90), ("Nakuru", 155)],
    "Kapsabet": [("Eldoret", 40), ("Kisumu", 120)],
    "Webuye": [("Eldoret", 75), ("Kakamega", 40)],
    "Kakamega": [("Webuye", 40), ("Kisumu", 55)],
    "Kisumu": [("Kakamega", 55), ("Kapsabet", 120), ("Kericho", 100), ("Kisii", 115)],
}

# General operational & safety rules for the self-driving trailer
road_rules_kb = {
    "Clear": "Maintain safe highway cruise speed and follow lane discipline.",
    "Fog": "Activate fog lights, reduce speed, and increase following distance.",
    "Potholes": "Slow down to protect trailer suspension and cargo integrity.",
    "Traffic Jam": "Hold position or evaluate alternate route if delay exceeds threshold.",
    "Roadblock": "Stop safely at a safe distance and reroute via knowledge base.",
    "Heavy Rain": "Reduce speed, engage wipers/headlights, and avoid harsh braking.",
}

## Step 5: Add New Facts to the Knowledge Base `[4 marks]`

In [2]:
def show_kb():
    print("\n--- Road Condition Rules KB ---")
    for cond, action in road_rules_kb.items():
        print(f"{cond:16} -> {action}")

    print("\n--- Sample Highway Connections KB ---")
    for town in ["Nairobi", "Naivasha", "Nakuru", "Molo", "Kericho"]:
        print(f"{town:10} -> {highway_kb.get(town, [])}")
    print("-------------------------------------")


def add_road_fact(origin, destination, distance_km):
    """Add or update a directed road fact in the highway KB."""
    highway_kb.setdefault(origin, [])
    highway_kb[origin] = [(nxt, d) for nxt, d in highway_kb[origin] if nxt != destination]
    highway_kb[origin].append((destination, distance_km))
    print(f"Added road fact: {origin} -> {destination} ({distance_km} km)")


def add_condition_fact(condition, action):
    """Add or update a road condition rule in the KB."""
    road_rules_kb[condition.title()] = action
    print(f"Added condition fact: '{condition.title()}' -> '{action}'")


def fetch_external_road_advisories(source_records=None):
    """Optional hook to ingest facts from an external API or scraper."""
    if not source_records:
        print("No external API/scraper records provided; using local KB facts.")
        return
    for rec in source_records:
        if {"origin", "destination", "distance_km"} <= rec.keys():
            add_road_fact(rec["origin"], rec["destination"], rec["distance_km"])
        elif {"condition", "action"} <= rec.keys():
            add_condition_fact(rec["condition"], rec["action"])


# Programmatic fact additions
add_condition_fact("Escarpment Wind", "Reduce speed and stabilize steering on steep/exposed sections.")
fetch_external_road_advisories()
show_kb()

Added condition fact: 'Escarpment Wind' -> 'Reduce speed and stabilize steering on steep/exposed sections.'
No external API/scraper records provided; using local KB facts.

--- Road Condition Rules KB ---
Clear            -> Maintain safe highway cruise speed and follow lane discipline.
Fog              -> Activate fog lights, reduce speed, and increase following distance.
Potholes         -> Slow down to protect trailer suspension and cargo integrity.
Traffic Jam      -> Hold position or evaluate alternate route if delay exceeds threshold.
Roadblock        -> Stop safely at a safe distance and reroute via knowledge base.
Heavy Rain       -> Reduce speed, engage wipers/headlights, and avoid harsh braking.
Escarpment Wind  -> Reduce speed and stabilize steering on steep/exposed sections.

--- Sample Highway Connections KB ---
Nairobi    -> [('Ruiru', 20), ('Naivasha', 90)]
Naivasha   -> [('Nakuru', 65), ('Narok', 100), ('Nairobi', 90)]
Nakuru     -> [('Naivasha', 65), ('Nyahururu', 105)

## Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers `[3 marks]`

In [3]:
def ucs_shortest_path(graph, start, goal):
    """Uniform Cost Search returning (optimal_path, total_cost_km)."""
    frontier = [(0, start, [start])]
    explored = set()
    best_cost = {start: 0}

    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored:
            continue
        if node == goal:
            return path, cost

        explored.add(node)
        for nxt, weight in graph.get(node, []):
            new_cost = cost + weight
            if nxt not in explored and (nxt not in best_cost or new_cost < best_cost[nxt]):
                best_cost[nxt] = new_cost
                heapq.heappush(frontier, (new_cost, nxt, path + [nxt]))

    return None, float("inf")


def query_trailer_agent(current_town="Nairobi", goal_town="Kisumu", road_condition="Clear"):
    """Pose a query to the KB for route planning and immediate driving advice."""
    start = current_town.strip().title()
    goal = goal_town.strip().title()
    cond = road_condition.strip().title()

    goal_reached = (start == goal)
    advice = road_rules_kb.get(cond, "Unknown condition — decelerate and apply caution.")
    path, cost = ucs_shortest_path(highway_kb, start, goal)

    print("=== Self-Driving Trailer KB Query Result ===")
    print(f"Current Location : {start}")
    print(f"Destination      : {goal}")
    print(f"Goal Test Passed : {goal_reached}")
    print(f"Road Condition   : {cond}")
    print(f"Driving Advice   : {advice}")
    if path:
        print(f"Optimal Route    : {' -> '.join(path)} ({cost} km)")
    else:
        print("Optimal Route    : No route found")


# Step 6 & 7 execution: Trailer starting in Nairobi heading to Kisumu in Fog
query_trailer_agent(current_town="Nairobi", goal_town="Kisumu", road_condition="Fog")

=== Self-Driving Trailer KB Query Result ===
Current Location : Nairobi
Destination      : Kisumu
Goal Test Passed : False
Road Condition   : Fog
Driving Advice   : Activate fog lights, reduce speed, and increase following distance.
Optimal Route    : Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (370 km)


## Step 8: Querying the Knowledge Base

In [4]:
# Query across multiple towns and road conditions
for town, cond in [
    ("Nairobi", "Clear"),
    ("Nakuru", "Potholes"),
    ("Kericho", "Heavy Rain"),
    ("Kisumu", "Clear"),
]:
    query_trailer_agent(town, "Kisumu", cond)
    print()

=== Self-Driving Trailer KB Query Result ===
Current Location : Nairobi
Destination      : Kisumu
Goal Test Passed : False
Road Condition   : Clear
Driving Advice   : Maintain safe highway cruise speed and follow lane discipline.
Optimal Route    : Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu (370 km)

=== Self-Driving Trailer KB Query Result ===
Current Location : Nakuru
Destination      : Kisumu
Goal Test Passed : False
Road Condition   : Potholes
Driving Advice   : Slow down to protect trailer suspension and cargo integrity.
Optimal Route    : Nakuru -> Molo -> Kericho -> Kisumu (215 km)

=== Self-Driving Trailer KB Query Result ===
Current Location : Kericho
Destination      : Kisumu
Goal Test Passed : False
Road Condition   : Heavy Rain
Driving Advice   : Reduce speed, engage wipers/headlights, and avoid harsh braking.
Optimal Route    : Kericho -> Kisumu (100 km)

=== Self-Driving Trailer KB Query Result ===
Current Location : Kisumu
Destination      : Kisumu
Goal T

## Step 9: Debugging the Knowledge Base `[2 marks]`

In [5]:
# Test edge cases, unknown conditions, and unreachable/disconnected towns
highway_kb["GhostTown"] = []  # Dead-end node test

debug_queries = [
    ("Nairobi", "Kisumu", "Clear"),
    ("Nairobi", "Kisumu", "Slippery Surface"),  # Unknown condition fallback
    ("GhostTown", "Kisumu", "Clear"),           # Unreachable goal test
    ("Kisumu", "Kisumu", "Clear"),              # Already at goal test
]

print("--- Debugging the Trailer Knowledge Base ---")
for start_t, goal_t, cond in debug_queries:
    path, cost = ucs_shortest_path(highway_kb, start_t, goal_t)
    action = road_rules_kb.get(cond, "Unknown condition — decelerate and apply caution.")
    print(f"From={start_t:10} To={goal_t:7} Cond={cond:16} -> Path={path}, Cost={cost}, Action={action}")

--- Debugging the Trailer Knowledge Base ---
From=Nairobi    To=Kisumu  Cond=Clear            -> Path=['Nairobi', 'Naivasha', 'Nakuru', 'Molo', 'Kericho', 'Kisumu'], Cost=370, Action=Maintain safe highway cruise speed and follow lane discipline.
From=Nairobi    To=Kisumu  Cond=Slippery Surface -> Path=['Nairobi', 'Naivasha', 'Nakuru', 'Molo', 'Kericho', 'Kisumu'], Cost=370, Action=Unknown condition — decelerate and apply caution.
From=GhostTown  To=Kisumu  Cond=Clear            -> Path=None, Cost=inf, Action=Maintain safe highway cruise speed and follow lane discipline.
From=Kisumu     To=Kisumu  Cond=Clear            -> Path=['Kisumu'], Cost=0, Action=Maintain safe highway cruise speed and follow lane discipline.


## Step 10: Final Reflection (Challenges & Recommended Improvements) `[3 marks]`

* **Challenges:**
  * **Static Map vs. Dynamic Road:** A fixed distance table treats roads as static and deterministic, whereas real Kenyan highways experience congestion, weather shifts, and roadwork delays.
  * **Heuristic Blindness in UCS:** Uniform Cost Search guarantees optimality on non-negative edge costs, but explores lower-cost branches first without directional guidance toward Kisumu.
  * **Incomplete Knowledge:** Unlisted conditions (e.g., `"Slippery Surface"`) fall back to generic caution rules unless explicitly added to the KB.
* **Recommended Improvements:**
  * Upgrade from UCS to **A* Search** using straight-line/road distance heuristics toward Kisumu to reduce unnecessary node expansions.
  * Integrate real-time traffic, elevation/grade (important for heavy trailers), and weather APIs into `fetch_external_road_advisories()` so edge weights reflect travel time or fuel/risk cost rather than distance alone.
  * Add structured rule validation (e.g., preventing negative edge weights or duplicate conflicting rules) during KB updates.